# CNN archutecture testing

In this notebook we constructed a basic ConvBlock for further use in the experimenmts.

In [1]:
import torch
import wandb
import time
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from src.cnn_utils import get_dataset

In [ ]:
# testing if waandb is working --> at thge beginning we had issues with the logging
print(wandb)
print(wandb.__file__)

<module 'wandb' from 'c:\\Users\\lucas\\Documents\\MSE\\4_Semester\\FTP_DeLearn\\venv_DeLearn\\Lib\\site-packages\\wandb\\__init__.py'>
c:\Users\lucas\Documents\MSE\4_Semester\FTP_DeLearn\venv_DeLearn\Lib\site-packages\wandb\__init__.py


In [10]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

cuda


In [2]:
# importing the dataset
train_dataset, val_dataset = get_dataset()

In [3]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


### Create Training loop for models

In [8]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_val = []
    acc_hist = []
    acc_hist_val = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)
    
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=False)
    
    if use_wandb:
        wandb.init(
            entity=entity,
            project=project,
            name=run_name,
            settings=wandb.Settings(init_timeout=300),
            config={
                "epochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        start = time.perf_counter()

        model.train_model()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        cost /= nbatches
        acc /= size

        model.eval()
        size_val = len(val_loader.dataset)
        nbatches_val = len(val_loader)
        cost_val, acc_val = 0.0, 0.0     

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_val += cost_ce(pred, Y).item()
                acc_val += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_val /= nbatches_val
        acc_val /= size_val

        end = time.perf_counter()
        epoch_time = end - start

        print(f"Epoch {epoch}: Train cost: {round(cost, 4)}, accuracy: {round(acc, 4)}, Validation cost: {round(cost_val, 4)}, accuracy: {round(acc_val, 4)} (Time: {round(epoch_time, 1)} seconds)")

        cost_hist.append(cost)
        cost_hist_val.append(cost_val)
        acc_hist.append(acc)
        acc_hist_val.append(acc_val)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_val,
                "val_accuracy": acc_val,
                "lr": optimizer.param_groups[0]['lr'],
                "epoch_time": epoch_time
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_val, acc_hist, acc_hist_val

### Creating one layer CNN-model

In [7]:
# creat a simple model with one convolutional layer and two fully connected layers

class first_model(nn.Module):
    
    def __init__(self, units=128):
        super(first_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [8]:
# create an model and its summary

model = first_model(128) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
              ReLU-6                  [-1, 128]               0
            Linear-7                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


Initiate Training

In [ ]:
# Default values for testing the convBlock architecture
batch_size = 64
nepochs = 50
lr = 0.001
units = 128

model = first_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e30_lr0.001_u128_1l_no-reg', use_wandb=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0: Train cost: 2.2489, accuracy: 0.1636, Validation cost: 2.2132, accuracy: 0.2133 (Time: 31.1288 seconds)
Epoch 1: Train cost: 2.1767, accuracy: 0.2215, Validation cost: 2.1713, accuracy: 0.2145 (Time: 30.9657 seconds)
Epoch 2: Train cost: 2.1237, accuracy: 0.2533, Validation cost: 2.1077, accuracy: 0.2812 (Time: 29.2917 seconds)
Epoch 3: Train cost: 2.0723, accuracy: 0.2845, Validation cost: 2.0611, accuracy: 0.2913 (Time: 29.2583 seconds)
Epoch 4: Train cost: 2.0185, accuracy: 0.3065, Validation cost: 2.0244, accuracy: 0.3118 (Time: 29.1928 seconds)
Epoch 5: Train cost: 1.9642, accuracy: 0.3252, Validation cost: 1.9604, accuracy: 0.3362 (Time: 29.3365 seconds)
Epoch 6: Train cost: 1.9164, accuracy: 0.342, Validation cost: 1.9475, accuracy: 0.3243 (Time: 29.2958 seconds)
Epoch 7: Train cost: 1.8756, accuracy: 0.3522, Validation cost: 1.8855, accuracy: 0.339 (Time: 29.6718 seconds)
Epoch 8: Train cost: 1.8402, accuracy: 0.365, Validation cost: 1.8645, accuracy: 0.3542 (Time: 29.

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,█▇▁▁▁▁▃▂▂▃▁▂▂▂▁▂▂▃▂▁▂▂▂▁▂▁▂▂▂▂▂▂▁▂▄█▇██▇
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
train_loss,██▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_accuracy,▁▁▃▄▅▅▆▅▆▅▆▆▆▆▇▇▇▇▇▇█▇▇█▇▇█▇██▇███████▇█
val_loss,██▇▆▅▄▄▄▃▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁
epoch,50
epoch_time,31.10418
lr,0.001
train_accuracy,0.61158


## Interpretation:
This first model is heavily in a overfitting regime.\
Really bad vallidation accuracy and loss.\
The Result was kind of expected looking at the amount of parameters that are estimted during training (+50 Mio).\
Therefore we tried to construct a ConvBlock that performs better.
Possibilities to improve the model:
- heavier downsampling before passing into a fully connected layer
- reduce/ make the dense layer smaller (less units)
- add regularization:
    - Dropout
    - better optimizer
    - early stopping
    - etc.

In the next step dropout with p=0.5 is added before the linear-layer.

In [7]:
# creat a simple model with one convolutional layer and two fully connected layers

class improved_model(nn.Module):
    
    def __init__(self, units=128, drop=0.5):
        super(improved_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), # Conv with 32 filters, kernel size 3x3 and padding 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [8]:
model = improved_model(128, 0.5) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
              ReLU-6                  [-1, 128]               0
           Dropout-7                  [-1, 128]               0
            Linear-8                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


In [9]:
batch_size = 64
nepochs = 50
lr = 0.01
units = 128
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter

model = improved_model(units, drop=0.5)
optimizer = torch.optim.Adam(params=model.parameters(), lr = lr, weight_decay=wd)
cost_train_adam, cost_valid_adam, acc_train_adam, acc_valid_adam = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e50_lr0.01_u128_1l_adam_drop0.5', use_wandb=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0: Train cost: 3.766, accuracy: 0.2046, Validation cost: 1.9983, accuracy: 0.259 (Time: 36.7 seconds)
Epoch 1: Train cost: 2.0274, accuracy: 0.2386, Validation cost: 1.9698, accuracy: 0.2703 (Time: 31.6 seconds)
Epoch 2: Train cost: 2.0013, accuracy: 0.2591, Validation cost: 1.9624, accuracy: 0.2762 (Time: 31.6 seconds)
Epoch 3: Train cost: 1.9873, accuracy: 0.263, Validation cost: 1.964, accuracy: 0.2852 (Time: 32.0 seconds)
Epoch 4: Train cost: 1.9731, accuracy: 0.2668, Validation cost: 1.9487, accuracy: 0.2883 (Time: 32.3 seconds)
Epoch 5: Train cost: 1.9656, accuracy: 0.2727, Validation cost: 1.9496, accuracy: 0.2745 (Time: 31.6 seconds)
Epoch 6: Train cost: 1.9553, accuracy: 0.2757, Validation cost: 1.9416, accuracy: 0.2983 (Time: 31.6 seconds)
Epoch 7: Train cost: 1.9417, accuracy: 0.2843, Validation cost: 1.9453, accuracy: 0.2945 (Time: 32.2 seconds)
Epoch 8: Train cost: 1.9348, accuracy: 0.2863, Validation cost: 1.9403, accuracy: 0.2897 (Time: 31.9 seconds)
Epoch 9: Train

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,█▂▂▃▃▂▃▃▂▂▃▃▂▃▃▂▁▁▁▁▁▁▂▁▁▂▂▂▃▂▁▁▁▁▁▁▁▂▂▁
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▄▅▆▆▆▇▇▇▇▇██████▂▁▂▄▆▆▅▇▇▇▇▄▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▃▃▂▁▄▂▁▁▆▁▁█▃▃▃▃▃▃▃▃▃▅
val_accuracy,▇▇▇██████▇██████▁▁▃▅▇▁▇▇▇▇▇▇▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
epoch_time,30.7529
lr,0.01
train_accuracy,0.09967


After testing with different learning rates and still a huge problem with too many parameters in the model. We decided to make a small change in architecture.\
By adding the AdaptiveAvgPool2d the amount of parameter is reduced from ~51 mio to 6'410.\
This setup is better suited to prevent overfitting.

In [10]:
class improved_model(nn.Module):
    
    def __init__(self, units=128, drop=0.5):
        super(improved_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), # Conv with 32 filters, kernel size 3x3 and padding 
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)), # global average pooling to reduce the spatial dimensions to 1x1, resulting in a feature vector of size 32
            nn.Flatten(),
            nn.Linear(32,units),
            nn.ReLU(),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [11]:
model = improved_model(128, 0.5) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
 AdaptiveAvgPool2d-3             [-1, 32, 1, 1]               0
           Flatten-4                   [-1, 32]               0
            Linear-5                  [-1, 128]           4,224
              ReLU-6                  [-1, 128]               0
           Dropout-7                  [-1, 128]               0
            Linear-8                   [-1, 10]           1,290
Total params: 6,410
Trainable params: 6,410
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 24.50
Params size (MB): 0.02
Estimated Total Size (MB): 25.10
----------------------------------------------------------------


This model with the same learning rate performs worse but is way less prone to overfiting.
Since there only is one layer in the model and with the new architecture less parameters for the model to learn the resulting accuracy is worse, which is expected.\
Visualization can ba found in the CNN-Architecture report from W&B. 

In [12]:
batch_size = 64 # default batch size, can be tuned as a hyperparameter.
nepochs = 50
units = 128 # default number of units in the fully connected layer, can be tuned as a hyperparameter
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter
lrs = [0.1, 0.01, 0.001, 0.0001]

results = {}

for lr in lrs:
    print(f"\n=== Training with lr = {lr} ===")
    
    # re-initialize model every time!
    model = improved_model(units, drop=0.5)
    
    # using Adam optimizer
    optimizer = torch.optim.Adam(
        params=model.parameters(),
        lr=lr,
        weight_decay=wd
    )
    
    run_name = f"CNN_bs{batch_size}_e{nepochs}_lr_{lr}_u{units}_drop0.5"
    
    cost_train, cost_valid, acc_train, acc_valid = train_eval(
        model,
        optimizer,
        nepochs,
        batch_size,
        train_dataset,
        val_dataset,
        device,
        entity='MSE_DeLearn_SPR26',
        project='MPW-CNN',
        run_name=run_name,
        use_wandb=True
    )
    
    results[lr] = {
        "train_loss": cost_train,
        "val_loss": cost_valid,
        "train_acc": acc_train,
        "val_acc": acc_valid
    }


=== Training with lr = 0.1 ===


Epoch 0: Train cost: 2.321, accuracy: 0.0992, Validation cost: 2.3178, accuracy: 0.1 (Time: 30.1 seconds)
Epoch 1: Train cost: 2.3134, accuracy: 0.0987, Validation cost: 2.3099, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 2: Train cost: 2.3111, accuracy: 0.1006, Validation cost: 2.3182, accuracy: 0.1 (Time: 30.7 seconds)
Epoch 3: Train cost: 2.3139, accuracy: 0.1019, Validation cost: 2.3059, accuracy: 0.1 (Time: 31.0 seconds)
Epoch 4: Train cost: 2.3127, accuracy: 0.0998, Validation cost: 2.3233, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 5: Train cost: 2.3124, accuracy: 0.0998, Validation cost: 2.3117, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 6: Train cost: 2.3134, accuracy: 0.0997, Validation cost: 2.3119, accuracy: 0.1 (Time: 30.5 seconds)
Epoch 7: Train cost: 2.3149, accuracy: 0.0982, Validation cost: 2.3155, accuracy: 0.1 (Time: 30.7 seconds)
Epoch 8: Train cost: 2.3135, accuracy: 0.0987, Validation cost: 2.3097, accuracy: 0.1 (Time: 30.8 seconds)
Epoch 9: Train cost: 2.3137, accuracy:

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,▆█▇██▇▇▇▄▃▂██▄▂▁▂▁▁▂▂▂▁▂▁▁▂▂▂▂▇▂▂▂▂▂▂▁▁█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▄▃▅▇▄▄▃▃▅▂▆▅▃▄▂▄▃▄▅▂█▅▁▅▃▅▄▅▇▂▂▅▆▅▁█▂▅▁▇
train_loss,▃▂▁▂▁▂▂▂▂▅▄▁▂▅▂█▃▁▁▂▁▁▁▁▃▁▁▄▁▂▂▁▂▂▁▂▂▁▅▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▆▃▆▂█▄▅▃▅▅▄▃▁▇▄▅▁▄▄▂▄▄▆▂▄▃▂▅▃▅▂▁▄▅▄█▁▅▄▄
epoch,50
epoch_time,30.85268
lr,0.1
train_accuracy,0.10237



=== Training with lr = 0.01 ===


Epoch 0: Train cost: 2.285, accuracy: 0.1249, Validation cost: 2.2448, accuracy: 0.1587 (Time: 30.8 seconds)
Epoch 1: Train cost: 2.2364, accuracy: 0.1598, Validation cost: 2.2105, accuracy: 0.1782 (Time: 30.7 seconds)
Epoch 2: Train cost: 2.2198, accuracy: 0.1704, Validation cost: 2.1994, accuracy: 0.1838 (Time: 29.0 seconds)
Epoch 3: Train cost: 2.198, accuracy: 0.1811, Validation cost: 2.1169, accuracy: 0.2413 (Time: 28.5 seconds)
Epoch 4: Train cost: 2.1045, accuracy: 0.2307, Validation cost: 2.0534, accuracy: 0.2575 (Time: 28.4 seconds)
Epoch 5: Train cost: 2.068, accuracy: 0.2481, Validation cost: 2.0122, accuracy: 0.27 (Time: 28.8 seconds)
Epoch 6: Train cost: 2.0501, accuracy: 0.2563, Validation cost: 1.9996, accuracy: 0.2772 (Time: 30.9 seconds)
Epoch 7: Train cost: 2.0437, accuracy: 0.2578, Validation cost: 1.992, accuracy: 0.2708 (Time: 29.6 seconds)
Epoch 8: Train cost: 2.0415, accuracy: 0.2575, Validation cost: 1.9869, accuracy: 0.2795 (Time: 28.7 seconds)
Epoch 9: Train c

...

Epoch 0: Train cost: 2.2966, accuracy: 0.1165, Validation cost: 2.2801, accuracy: 0.1553 (Time: 31.0 seconds)
Epoch 1: Train cost: 2.2654, accuracy: 0.1492, Validation cost: 2.2498, accuracy: 0.1582 (Time: 31.1 seconds)
Epoch 2: Train cost: 2.2394, accuracy: 0.1658, Validation cost: 2.2298, accuracy: 0.1773 (Time: 30.3 seconds)
Epoch 3: Train cost: 2.22, accuracy: 0.1815, Validation cost: 2.2099, accuracy: 0.1875 (Time: 28.4 seconds)
Epoch 4: Train cost: 2.1923, accuracy: 0.2061, Validation cost: 2.1579, accuracy: 0.2268 (Time: 28.6 seconds)
Epoch 5: Train cost: 2.1437, accuracy: 0.225, Validation cost: 2.1073, accuracy: 0.252 (Time: 28.5 seconds)
Epoch 6: Train cost: 2.1037, accuracy: 0.2422, Validation cost: 2.0759, accuracy: 0.2595 (Time: 28.3 seconds)
Epoch 7: Train cost: 2.0787, accuracy: 0.25, Validation cost: 2.0545, accuracy: 0.268 (Time: 28.1 seconds)
Epoch 8: Train cost: 2.0622, accuracy: 0.2597, Validation cost: 2.0429, accuracy: 0.2715 (Time: 28.2 seconds)
Epoch 9: Train co

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,██▆▂▃▂▁▁▁▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▁▁▁▂▂▂▂▂▂
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▃▃▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████████
train_loss,█▇▇▆▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▂▂▃▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███▇██████████
val_loss,██▇▇▆▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,50
epoch_time,28.29033
lr,0.001
train_accuracy,0.33654



=== Training with lr = 0.0001 ===


Epoch 0: Train cost: 2.3044, accuracy: 0.1024, Validation cost: 2.3014, accuracy: 0.1295 (Time: 28.2 seconds)
Epoch 1: Train cost: 2.3014, accuracy: 0.1094, Validation cost: 2.2993, accuracy: 0.1137 (Time: 28.1 seconds)
Epoch 2: Train cost: 2.2999, accuracy: 0.1108, Validation cost: 2.2972, accuracy: 0.1337 (Time: 28.5 seconds)
Epoch 3: Train cost: 2.2968, accuracy: 0.1241, Validation cost: 2.2947, accuracy: 0.143 (Time: 28.2 seconds)
Epoch 4: Train cost: 2.2949, accuracy: 0.1267, Validation cost: 2.2916, accuracy: 0.1333 (Time: 28.1 seconds)
Epoch 5: Train cost: 2.2916, accuracy: 0.128, Validation cost: 2.2879, accuracy: 0.1387 (Time: 28.2 seconds)
Epoch 6: Train cost: 2.2869, accuracy: 0.1345, Validation cost: 2.2836, accuracy: 0.1383 (Time: 28.1 seconds)
Epoch 7: Train cost: 2.2834, accuracy: 0.1357, Validation cost: 2.2795, accuracy: 0.1395 (Time: 28.1 seconds)
Epoch 8: Train cost: 2.2786, accuracy: 0.1408, Validation cost: 2.2754, accuracy: 0.1412 (Time: 28.0 seconds)
Epoch 9: Tra

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch_time,▅▃█▄▃▃▁▁▂▅▄▄▃▂▁▃▃▁▂▃▇▅▄▄▃▅▃▇▄▆▆▄▅▄▄▅▆▆▄▄
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
train_loss,███▇▇▇▇▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
val_accuracy,▂▁▃▃▃▃▃▃▃▃▄▄▄▄▅▄▄▅▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
val_loss,████▇▇▇▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁
epoch,50
epoch_time,28.15384
lr,0.0001
train_accuracy,0.19925


In [4]:
class improved_model(nn.Module):
    
    def __init__(self, drop=0.5):
        super(improved_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), # Conv with 32 filters, kernel size 3x3 and padding 
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)), # global average pooling to reduce the spatial dimensions to 1x1, resulting in a feature vector of size 32
            nn.Flatten(),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.Linear(32,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [6]:
model = improved_model(0.5) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
 AdaptiveAvgPool2d-3             [-1, 32, 1, 1]               0
           Flatten-4                   [-1, 32]               0
           Dropout-5                   [-1, 32]               0
            Linear-6                   [-1, 10]             330
Total params: 1,226
Trainable params: 1,226
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 24.50
Params size (MB): 0.00
Estimated Total Size (MB): 25.08
----------------------------------------------------------------


In [11]:
batch_size = 64
nepochs = 50
lr = 0.001

wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter

model = improved_model(drop=0.5)
optimizer = torch.optim.Adam(params=model.parameters(), lr = lr, weight_decay=wd)
cost_train_adam, cost_valid_adam, acc_train_adam, acc_valid_adam = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e50_lr0.001_no-Lin-layer_1l_adam_drop0.5', use_wandb=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0: Train cost: 2.3019, accuracy: 0.1103, Validation cost: 2.2948, accuracy: 0.1372 (Time: 31.5 seconds)
Epoch 1: Train cost: 2.2932, accuracy: 0.1201, Validation cost: 2.2859, accuracy: 0.1455 (Time: 30.2 seconds)
Epoch 2: Train cost: 2.2858, accuracy: 0.1305, Validation cost: 2.278, accuracy: 0.1425 (Time: 30.4 seconds)
Epoch 3: Train cost: 2.2759, accuracy: 0.1399, Validation cost: 2.2681, accuracy: 0.1458 (Time: 30.5 seconds)
Epoch 4: Train cost: 2.2682, accuracy: 0.149, Validation cost: 2.2601, accuracy: 0.18 (Time: 30.3 seconds)
Epoch 5: Train cost: 2.2601, accuracy: 0.1571, Validation cost: 2.2499, accuracy: 0.1755 (Time: 30.2 seconds)
Epoch 6: Train cost: 2.2529, accuracy: 0.1655, Validation cost: 2.2446, accuracy: 0.1902 (Time: 30.4 seconds)
Epoch 7: Train cost: 2.2435, accuracy: 0.177, Validation cost: 2.2343, accuracy: 0.2 (Time: 30.2 seconds)
Epoch 8: Train cost: 2.238, accuracy: 0.1818, Validation cost: 2.2267, accuracy: 0.2092 (Time: 30.7 seconds)
Epoch 9: Train cost

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch_time,█▃▄▄▄▄▃▅▃▅▄▄▄▄▁▄▄▄▅▅▄▅▅▅▅▅▆▅▆▅▄▅▅▄▄▆▅▅▄▄
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇█▇█▇████████
train_loss,██▇▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▁▃▄▅▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇████▇█▇███▇█
val_loss,██▇▇▇▆▆▆▆▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch,50
epoch_time,30.36111
lr,0.001
train_accuracy,0.24592
